# LLM4Teach — Kaggle Training Notebook

**Symbolic RL with Qwen2.5 planner + reflection memory**

### Before running:
1. Sidebar → **Add Data → Your Datasets → `zip1231`**
2. Sidebar → **Settings → Internet: ON**
3. Sidebar → **Accelerator: GPU T4**
4. Run cells **top to bottom in order**

| Cell | What it does | Time |
|------|-------------|------|
| 1 | Copy project to writable directory | 30s |
| 2 | Fix NumPy + install missing libraries | 2 min |
| 3 | Install Ollama (with zstd fix) | 1 min |
| 4 | Start Ollama server | 10s |
| 5 | Pull Qwen model | ~5 min |
| 6 | Sanity checks | 30s |
| 7 | **Run training** | hours |
| 8 | Inspect logs | instant |
| 9 | **Zip and save all logs** | instant |
| 10 | Plot training curves | instant |

In [ ]:
# ── Cell 0: CONFIG — run this first, always ──────────────────────────────────
# This cell defines all paths and settings used by every other cell.
# Run this cell FIRST every time — even after interrupting training.
# Cells 8, 9, 10 (inspect / zip / plot) only need this cell to work.

import os, sys

# ── Training config (edit here) ───────────────────────────────────────────────
DATASET_NAME  = 'zip1231'       # your Kaggle dataset slug
TASK          = 'SimpleDoorKey' # SimpleDoorKey | LavaDoorKey | ColoredDoorKey | TwoDoor
SAVEDIR       = 'kaggle_run1'   # must match --savedir in Cell 7
N_ITR         = 50              # training iterations
TRAJ_PER_ITR  = 10              # episodes per iteration
LLM_MODEL     = 'qwen2.5:3b'   # planner model
REFLECT_MODEL = 'qwen2.5:3b'   # reflector model
# ─────────────────────────────────────────────────────────────────────────────

# Derived paths (do not change)
DEST    = '/kaggle/working/LLM4Teach-main'
LOG_DIR = os.path.join(DEST, 'log', 'ppo', TASK, SAVEDIR + '-0')

# Add project to Python path if already copied
if os.path.exists(DEST) and DEST not in sys.path:
    sys.path.insert(0, DEST)
    os.chdir(DEST)

print('Config loaded.')
print(f'  TASK     : {TASK}')
print(f'  SAVEDIR  : {SAVEDIR}')
print(f'  LOG_DIR  : {LOG_DIR}')
print(f'  DEST     : {DEST}')
print()

# Check whether log files already exist (useful after an interrupted run)
log_files = ['training.log', 'trajectories.jsonl', 'metrics.csv',
             'prompts.log', 'debug.log']
if os.path.isdir(LOG_DIR):
    print('Existing log files:')
    for fname in log_files:
        fpath = os.path.join(LOG_DIR, fname)
        if os.path.exists(fpath):
            kb    = os.path.getsize(fpath) / 1024
            lines = sum(1 for _ in open(fpath, encoding='utf-8', errors='ignore'))
            print(f'  ✅ {fname:<25} {kb:7.1f} KB   {lines} lines')
        else:
            print(f'  ·  {fname:<25} not yet created')
else:
    print('Log directory does not exist yet — training has not started.')

In [ ]:
# ── Cell 1: Copy project to writable directory ────────────────────────────────
# /kaggle/input is read-only. Training writes logs so we copy to /kaggle/working first.

import os, sys, shutil

DATASET_NAME = 'zip1231'   # ← change if your Kaggle dataset slug is different
INPUT_ROOT   = f'/kaggle/input/{DATASET_NAME}/LLM4Teach-main'
DEST         = '/kaggle/working/LLM4Teach-main'

# Auto-detect folder name if default path doesn't exist
if not os.path.exists(INPUT_ROOT):
    base = f'/kaggle/input/{DATASET_NAME}'
    print('Dataset contents:', os.listdir(base))
    for item in os.listdir(base):
        candidate = os.path.join(base, item)
        if os.path.isdir(candidate) and os.path.exists(os.path.join(candidate, 'main.py')):
            INPUT_ROOT = candidate
            print(f'Found project at: {INPUT_ROOT}')
            break
    else:
        raise FileNotFoundError(
            f'main.py not found under /kaggle/input/{DATASET_NAME}/. '
            f'Update DATASET_NAME above.'
        )

if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(INPUT_ROOT, DEST)

os.chdir(DEST)
sys.path.insert(0, DEST)

print(f'Project root : {DEST}')
print(f'Python files : {[f for f in os.listdir(DEST) if f.endswith(".py")]}')

In [ ]:
# ── Cell 2: Fix NumPy + install missing libraries ─────────────────────────────
# Kaggle ships NumPy 2.x but pre-installed cv2 needs NumPy 1.x → downgrade first.
# PyTorch with CUDA is already installed by Kaggle — we don't reinstall it.

import subprocess, torch

print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))
    print('VRAM     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING  : No GPU — check Accelerator in sidebar')

print('\nFixing NumPy + OpenCV...')
subprocess.run('pip install -q "numpy<2"', shell=True)
subprocess.run('pip install -q --force-reinstall opencv-python-headless', shell=True)

print('Installing project dependencies...')
subprocess.run(
    'pip install -q minigrid==3.1.0 tensorboard==2.20.0 requests==2.31.0',
    shell=True
)

import numpy, cv2, gymnasium, minigrid, requests
print(f'numpy     : {numpy.__version__}')
print(f'cv2       : {cv2.__version__}')
print(f'gymnasium : {gymnasium.__version__}')
print(f'minigrid  : {minigrid.__version__}')
print('\n✅ Done. If NumPy version changed → Restart kernel then run all cells again.')

In [ ]:
# ── Cell 3: Install Ollama ────────────────────────────────────────────────────
# zstd must be installed first — required by the Ollama installer on Kaggle.

import subprocess

print('Installing zstd (required by Ollama installer)...')
subprocess.run('apt-get install -y zstd', shell=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('Installing Ollama...')
result = subprocess.run(
    'curl -fsSL https://ollama.ai/install.sh | sh',
    shell=True, capture_output=True, text=True
)
# Show last few lines only (installer is verbose)
for line in result.stdout.strip().splitlines()[-5:]:
    print(line)

v = subprocess.run('ollama --version', shell=True, capture_output=True, text=True)
print(f'\n✅ Ollama: {v.stdout.strip()}')

In [ ]:
# ── Cell 4: Start Ollama server ───────────────────────────────────────────────

import subprocess, time, requests as req

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT
)

for i in range(30):
    try:
        if req.get('http://localhost:11434/api/tags', timeout=2).status_code == 200:
            print(f'✅ Ollama server ready (took {i+1}s)')
            break
    except Exception:
        pass
    print(f'  Waiting... {i+1}/30', flush=True)
    time.sleep(1)
else:
    print('❌ Ollama did not start. Log:')
    print(open('/tmp/ollama.log').read()[-500:])

In [ ]:
# ── Cell 5: Pull Qwen model ───────────────────────────────────────────────────
# Using qwen2.5:3b for BOTH planner and reflector.
# os.system() streams download progress live in Kaggle.

import os, subprocess, time, requests as req

# Restart server if it died between cells
try:
    req.get('http://localhost:11434/api/tags', timeout=2)
except Exception:
    print('Restarting Ollama server...')
    subprocess.Popen(['ollama', 'serve'],
                     stdout=open('/tmp/ollama.log', 'w'),
                     stderr=subprocess.STDOUT)
    time.sleep(4)

print('Pulling qwen2.5:3b (~2 GB) — progress shown below...')
os.system('ollama pull qwen2.5:3b')

print('\nModels available:')
os.system('ollama list')

In [ ]:
# ── Cell 6: Sanity checks ─────────────────────────────────────────────────────

import os, sys, subprocess, time, requests as req

DEST = '/kaggle/working/LLM4Teach-main'
os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)

# Restart Ollama if needed
try:
    req.get('http://localhost:11434/api/tags', timeout=2)
except Exception:
    subprocess.Popen(['ollama', 'serve'],
                     stdout=open('/tmp/ollama.log', 'w'),
                     stderr=subprocess.STDOUT)
    time.sleep(4)

# Symbolic parser
from utils.symbolic_parser import validate_plan, strict_parse
assert validate_plan('go to <key>, pick up <key>, open <door>')[0]
assert strict_parse('go to <handle>') is None
print('✅ Symbolic parser')

# Reflection memory
from memory.memory_buffer import ReflectionMemory
m = ReflectionMemory(maxlen=5)
m.add_memory('Test.', success=False, episode_id=1)
print('✅ Reflection memory')

# Trajectory logger
from simulator.trajectory_logger import TrajectoryLogger, StepRecord, EpisodeRecord
print('✅ Trajectory logger')

# Qwen LLM
from utils.qwen_llm import QwenLLM
llm = QwenLLM(backend='ollama', model='qwen2.5:3b', temperature=0.1)
ok  = llm.verify_inference()
print(f'✅ qwen2.5:3b (planner)   → {"OK" if ok else "❌ FAILED"}')

from memory.reflection import QwenReflector
QwenReflector(backend='ollama', model='qwen2.5:3b')
print('✅ qwen2.5:3b (reflector) → OK')

print('\n' + '='*50)
print('All checks passed — ready to train!')
print('='*50)

In [ ]:
# ── Cell 7: Run Training ──────────────────────────────────────────────────────
# Config is set in Cell 0. Run Cell 0 first.
# Auto-zips logs immediately when training finishes or is interrupted.

import subprocess, sys, os, torch, time, zipfile, glob, requests as req

DEST   = '/kaggle/working/LLM4Teach-main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.chdir(DEST)

# Restart Ollama if it died between cells
try:
    req.get('http://localhost:11434/api/tags', timeout=2)
except Exception:
    print('Restarting Ollama...')
    subprocess.Popen(['ollama', 'serve'],
                     stdout=open('/tmp/ollama.log', 'w'),
                     stderr=subprocess.STDOUT)
    time.sleep(4)

print(f'Device   : {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'Task     : {TASK}')
print(f'Save dir : {SAVEDIR}')
print(f'Planner  : {LLM_MODEL}')
print(f'Reflector: {REFLECT_MODEL}')
print(f'Logs →   : {LOG_DIR}')

cmd = [
    sys.executable, 'main.py', 'train',
    '--task',               TASK,
    '--savedir',            SAVEDIR,
    '--n_itr',              str(N_ITR),
    '--traj_per_itr',       str(TRAJ_PER_ITR),
    '--device',             DEVICE,
    '--llm_backend',        'ollama',
    '--llm_model',          LLM_MODEL,
    '--reflection',
    '--reflection_backend', 'ollama',
    '--reflection_model',   REFLECT_MODEL,
]

print('\nCommand:', ' '.join(cmd))
print('=' * 60)

def _auto_zip(reason='finished'):
    """Zip all logs immediately — called after training ends or is interrupted."""
    if not os.path.isdir(LOG_DIR):
        print('No log directory found — nothing to zip.')
        return
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    zip_name  = f'/kaggle/working/llm4teach_{TASK}_{SAVEDIR}_{timestamp}.zip'
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in ['training.log', 'trajectories.jsonl', 'metrics.csv',
                      'prompts.log', 'debug.log']:
            fpath = os.path.join(LOG_DIR, fname)
            if os.path.exists(fpath):
                zf.write(fpath, arcname=os.path.join('logs', fname))
                print(f'  ✅ {fname}  ({os.path.getsize(fpath)/1024:.1f} KB)')
        for f in glob.glob(f'{LOG_DIR}/**/reflection_memory.json', recursive=True):
            zf.write(f, arcname=os.path.join('logs', 'reflection_memory.json'))
            print(f'  ✅ reflection_memory.json')
        for f in glob.glob(f'{LOG_DIR}/**/*.pt', recursive=True):
            zf.write(f, arcname=os.path.join('model', os.path.basename(f)))
            print(f'  ✅ {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)')
        for f in glob.glob(f'{LOG_DIR}/**/events.out.tfevents*', recursive=True):
            zf.write(f, arcname=os.path.join('tensorboard', os.path.basename(f)))
    size_mb = os.path.getsize(zip_name) / 1e6
    print(f'\n✅ Auto-zip ({reason}): {os.path.basename(zip_name)}  ({size_mb:.2f} MB)')
    print('Download from Output panel → right sidebar.')

# ── Run training ──────────────────────────────────────────────────────────────
try:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=DEST,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    print('=' * 60)
    print(f'Training finished. Exit code: {process.returncode}')

except KeyboardInterrupt:
    print('\n' + '=' * 60)
    print('Training interrupted.')
    try:
        process.terminate()
    except Exception:
        pass

finally:
    # ── Always zip — whether training finished, was interrupted, or crashed ──
    print('\nAuto-zipping logs...')
    _auto_zip(reason='auto')

In [ ]:
# ── Cell 8: Inspect logs ──────────────────────────────────────────────────────
# Works even if training was interrupted. Just needs Cell 0 to have been run.

import os, glob, json, pandas as pd

print(f'Log directory: {LOG_DIR}')
print()

if not os.path.isdir(LOG_DIR):
    print('Log directory does not exist yet.')
    print('Either training has not started, or TASK/SAVEDIR in Cell 0 is wrong.')
else:
    # ── File sizes ────────────────────────────────────────────────────────────
    print('── Log files ──')
    for fname in ['training.log', 'trajectories.jsonl', 'metrics.csv',
                  'prompts.log', 'debug.log']:
        fpath = os.path.join(LOG_DIR, fname)
        if os.path.exists(fpath):
            kb    = os.path.getsize(fpath) / 1024
            lines = sum(1 for _ in open(fpath, encoding='utf-8', errors='ignore'))
            print(f'  ✅ {fname:<25} {kb:7.1f} KB   {lines} lines')
        else:
            print(f'  ·  {fname:<25} not created yet')
    print()

    # ── metrics.csv summary ───────────────────────────────────────────────────
    csv_path = os.path.join(LOG_DIR, 'metrics.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f'── metrics.csv: {len(df)} episodes logged ──')
        print(df[['episode','reward','success','steps','llm_calls',
                  'interventions','failure_type']].tail(10).to_string(index=False))
        print(f'\nSuccess rate : {df["success"].mean():.1%}')
        print(f'Avg reward   : {df["reward"].mean():.4f}')
        print(f'Failure types:\n{df["failure_type"].value_counts().to_string()}')
        print()

    # ── Last 20 lines of training.log ─────────────────────────────────────────
    train_log = os.path.join(LOG_DIR, 'training.log')
    if os.path.exists(train_log):
        lines = open(train_log, encoding='utf-8', errors='ignore').readlines()
        print(f'── training.log (last 20 lines of {len(lines)} total) ──')
        print(''.join(lines[-20:]))

    # ── Saved models ──────────────────────────────────────────────────────────
    models = glob.glob(f'{LOG_DIR}/**/*.pt', recursive=True)
    if models:
        print('── Saved models ──')
        for f in models:
            print(f'  {f}  ({os.path.getsize(f)/1e6:.1f} MB)')

    # ── Reflection memory ─────────────────────────────────────────────────────
    for f in glob.glob(f'{LOG_DIR}/**/reflection_memory.json', recursive=True):
        data    = json.load(open(f))
        entries = data.get('entries', [])
        print(f'\n── Reflections: {len(entries)} stored ──')
        for e in entries[-3:]:
            tag = '✅' if e.get('success') else '❌'
            print(f'  {tag} Ep {e.get("episode_id","?")}: {str(e.get("reflection",""))[:90]}')

In [ ]:
# ── Cell 9: Zip all logs and save ─────────────────────────────────────────────
# Run this anytime — even after interrupting training.
# Only needs Cell 0 to have been run first.
# The ZIP is saved to /kaggle/working/ which Kaggle auto-saves.

import os, zipfile, glob, time

if not os.path.isdir(LOG_DIR):
    print(f'Log directory not found: {LOG_DIR}')
    print('Check TASK and SAVEDIR in Cell 0.')
else:
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    zip_name  = f'/kaggle/working/llm4teach_logs_{TASK}_{SAVEDIR}_{timestamp}.zip'

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:

        # Core log files
        for fname in ['training.log', 'trajectories.jsonl', 'metrics.csv',
                      'prompts.log', 'debug.log']:
            fpath = os.path.join(LOG_DIR, fname)
            if os.path.exists(fpath):
                zf.write(fpath, arcname=os.path.join('logs', fname))
                print(f'  ✅ {fname:<25} ({os.path.getsize(fpath)/1024:.1f} KB)')
            else:
                print(f'  ·  {fname:<25} skipped (not found)')

        # Reflection memory
        for f in glob.glob(f'{LOG_DIR}/**/reflection_memory.json', recursive=True):
            zf.write(f, arcname=os.path.join('logs', 'reflection_memory.json'))
            print(f'  ✅ reflection_memory.json')

        # Saved model checkpoints
        for f in glob.glob(f'{LOG_DIR}/**/*.pt', recursive=True):
            zf.write(f, arcname=os.path.join('model', os.path.basename(f)))
            print(f'  ✅ {os.path.basename(f):<25} ({os.path.getsize(f)/1e6:.1f} MB)')

        # TensorBoard event files
        for f in glob.glob(f'{LOG_DIR}/**/events.out.tfevents*', recursive=True):
            zf.write(f, arcname=os.path.join('tensorboard', os.path.basename(f)))
            print(f'  ✅ {os.path.basename(f)}')

    zip_size = os.path.getsize(zip_name) / 1e6
    print(f'\n✅ ZIP saved: {os.path.basename(zip_name)}')
    print(f'   Size     : {zip_size:.2f} MB')
    print(f'\nDownload → Output panel (right sidebar) → Output files.')

In [ ]:
# ── Cell 10: Plot training curves from metrics.csv ───────────────────────────
# Works with partial data — plot whatever episodes were logged so far.
# Only needs Cell 0 to have been run first.

import os, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

csv_path = os.path.join(LOG_DIR, 'metrics.csv')

if not os.path.exists(csv_path):
    print(f'metrics.csv not found at: {csv_path}')
    print('Training has not started or was interrupted before the first episode.')
else:
    df = pd.read_csv(csv_path)
    print(f'Plotting {len(df)} episodes from metrics.csv')

    def smooth(series, w=10):
        return series.rolling(window=w, min_periods=1).mean()

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(
        f'LLM4Teach — {TASK} / {SAVEDIR}  ({len(df)} episodes)',
        fontsize=13, fontweight='bold'
    )

    # Reward
    ax = axes[0, 0]
    ax.plot(df['episode'], df['reward'], alpha=0.3, color='steelblue', lw=0.8)
    ax.plot(df['episode'], smooth(df['reward']), color='steelblue', lw=2)
    ax.set_title('Episode Reward'); ax.set_xlabel('Episode'); ax.grid(alpha=0.3)

    # Success rate
    ax = axes[0, 1]
    ax.plot(df['episode'], smooth(df['success'], 20), color='green', lw=2)
    ax.set_ylim(0, 1.05); ax.set_title('Success Rate (smoothed)')
    ax.set_xlabel('Episode'); ax.grid(alpha=0.3)

    # LLM calls
    ax = axes[0, 2]
    ax.plot(df['episode'], smooth(df['llm_calls']), color='orange', lw=2)
    ax.set_title('LLM Calls / Episode'); ax.set_xlabel('Episode'); ax.grid(alpha=0.3)

    # Interventions
    ax = axes[1, 0]
    ax.plot(df['episode'], smooth(df['interventions']), color='red', lw=2)
    ax.set_title('Interventions / Episode'); ax.set_xlabel('Episode'); ax.grid(alpha=0.3)

    # PPO entropy
    ax = axes[1, 1]
    valid = df[df['avg_ppo_entropy'] >= 0]
    if len(valid):
        ax.plot(valid['episode'], smooth(valid['avg_ppo_entropy']), color='purple', lw=2)
    ax.set_title('PPO Entropy'); ax.set_xlabel('Episode'); ax.grid(alpha=0.3)

    # Failure breakdown
    ax = axes[1, 2]
    counts = df['failure_type'].value_counts()
    colors = ['#e74c3c','#e67e22','#3498db','#95a5a6','#2ecc71','#9b59b6']
    ax.bar(counts.index, counts.values, color=colors[:len(counts)])
    ax.set_title('Failure Attribution'); ax.set_xlabel('Type')
    ax.tick_params(axis='x', rotation=30); ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    out = '/kaggle/working/training_curves.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'\nSaved: {out}')
    print(f'Success rate : {df["success"].mean():.1%}')
    print(f'Avg reward   : {df["reward"].mean():.4f}')